In [ ]:
import os
import csv
import ast
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

from utils import savefig

plt.rcParams['font.size'] = 14

In [ ]:
def load_data(path, training_data_path, model_num, exp, paramters, training_data_length=200):
    data_single_model = {}
    with open(path/"contiguity_effect.csv", "r") as f:
        reader = csv.reader(f)
        for row in reader:
            data_single_model["model_num"] = model_num
            data_single_model["exp"] = exp

            for key, value in paramters.items():
                data_single_model[key] = value

            data_single_model["accuracy"] = float(row[0])
            data_single_model["forward_asymmetry"] = float(row[1])
            data_single_model["temporal_factor_old"] = float(row[2])
            data_single_model["temporal_factor"] = float(row[3])

            classifier_data = pickle.load(open(path/"ridge_classifier_stat.pkl", "rb"))
            data_single_model["index_decoding_accuracy_encoding_phase"] = classifier_data["index_enc_acc"]
            data_single_model["item_decoding_accuracy_encoding_phase"] = classifier_data["item_enc_acc"]
            data_single_model["last_item_decoding_accuracy_encoding_phase"] = classifier_data["item_enc_acc_last"]
            data_single_model["index_decoding_accuracy_recall_phase"] = classifier_data["index_rec_acc"]
            data_single_model["item_decoding_accuracy_recall_phase"] = classifier_data["item_rec_acc"]
            data_single_model["last_item_decoding_accuracy_recall_phase"] = classifier_data["item_rec_acc_last"]
            data_single_model["index_decoding_accuracy"] = (data_single_model["index_decoding_accuracy_encoding_phase"] + data_single_model["index_decoding_accuracy_recall_phase"]) / 2
            data_single_model["item_decoding_accuracy"] = (data_single_model["item_decoding_accuracy_encoding_phase"] + data_single_model["item_decoding_accuracy_recall_phase"]) / 2
            data_single_model["last_item_decoding_accuracy"] = (data_single_model["last_item_decoding_accuracy_encoding_phase"] + data_single_model["last_item_decoding_accuracy_recall_phase"]) / 2

            explained_variance_data = np.load(path/"explained_variance.npy")
            data_single_model["explained_variance_encoding_index"] = explained_variance_data[0]
            data_single_model["explained_variance_recall_index"] = explained_variance_data[1]
            data_single_model["explained_variance_index"] = (data_single_model["explained_variance_encoding_index"] + data_single_model["explained_variance_recall_index"]) / 2
            data_single_model["explained_variance_encoding_identity"] = explained_variance_data[2]
            data_single_model["explained_variance_recall_identity"] = explained_variance_data[3]
            data_single_model["explained_variance_identity"] = (data_single_model["explained_variance_encoding_identity"] + data_single_model["explained_variance_recall_identity"]) / 2

            cross_decoding_data = np.load(path/"cross_acc.npy")
            data_single_model["cross_decoding_accuracy_index_rec_enc"] = cross_decoding_data[0]
            data_single_model["cross_decoding_accuracy_identity_rec_enc"] = cross_decoding_data[1]
            data_single_model["cross_decoding_accuracy_index_enc_rec"] = cross_decoding_data[2]
            data_single_model["cross_decoding_accuracy_identity_enc_rec"] = cross_decoding_data[3]
            data_single_model["cross_decoding_accuracy_index"] = (data_single_model["cross_decoding_accuracy_index_rec_enc"] + data_single_model["cross_decoding_accuracy_index_enc_rec"]) / 2
            data_single_model["cross_decoding_accuracy_identity"] = (data_single_model["cross_decoding_accuracy_identity_rec_enc"] + data_single_model["cross_decoding_accuracy_identity_enc_rec"]) / 2

            try:
                training_data = np.load(training_data_path/"accuracy_2.npy")
            except:
                training_data = np.load(training_data_path/"accuracy_1.npy")
            # find the final consecutive zeros in the training data and set it to the last non-zero value
            zero_indices = np.where(training_data == 0)[0]
            if len(zero_indices) > 0:
                last_zero_index = zero_indices[0]
                training_data[last_zero_index:] = training_data[last_zero_index-1]
            if training_data.shape[0] < training_data_length and training_data.shape[0] > 0:
                training_data = np.pad(training_data, (0, training_data_length - training_data.shape[0]), mode='constant', constant_values=training_data[-1])
            data_single_model["training_accuracy"] = training_data

            f_recall_prob = open(path/"recall_probability.csv", "r")
            for row in f_recall_prob:
                data_single_model["crp_curve"] = np.array([float(x) for x in row.strip().split(",")])
                # print(data_single_model["crp_curve"].shape)
                break
            break
    return data_single_model


### load data

In [ ]:
gamma_names = ["0", "02", "04", "06", "08", "10"]
temporal_discount_factors = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
noise_levels = [0, 0.2, 0.4, 0.6, 0.8, 1]
noise_names = ["0", "02", "04", "06", "08", "1"]

In [ ]:
""" load vary gamma """

data = []

data_folder = Path("./experiments/VaryAllSeq8LargeNoise/figures/ValueMemoryGRU")
training_curve_folder = Path("./experiments/VaryAllSeq8LargeNoise/saved_models/ValueMemoryGRU")
perturbation_folder = Path("./experiments/VaryAllSeq8LargeNoise/figures/perturbation/ValueMemoryGRU")
item_invariance_folder = Path("./experiments/VaryAllSeq8LargeNoise/figures/item_invariant/ValueMemoryGRU")
time_invariance_folder = Path("./experiments/VaryAllSeq8LargeNoise/figures/time_invariant/ValueMemoryGRU")
seq_len = 8

exp = "main"

for gamma_name, temporal_discount_factor in zip(gamma_names, temporal_discount_factors):
    for noise_name, noise_level in zip(noise_names, noise_levels):
        setup_name = "setup_gamma{}_noise{}".format(gamma_name, noise_name)

        parameters = {
            "temporal_discount_factor": temporal_discount_factor,
            "noise_level": noise_level,
            "seq_len": seq_len,
            "wm_noise": 0.1
        }
        
        training_data_length = 200
        for i in range(20):
            data_path = data_folder / setup_name / str(i)
            training_data_path = training_curve_folder / (setup_name + "-" + str(i))
            if os.path.exists(data_path/"contiguity_effect.csv") and os.path.exists(data_path/"cross_acc.npy"):
                data_single_model = load_data(data_path, training_data_path, i, exp, parameters, training_data_length)
                if data_single_model:
                    data_single_model["pretrained"] = False
                    perturbation_data = np.load(perturbation_folder / setup_name / str(i) / "perturbation_accuracies.npy")
                    data_single_model["perturbation_accuracy"] = perturbation_data
                    if os.path.exists(item_invariance_folder / setup_name / str(i) / "decoding_accuracy.npy"):
                        item_invariance_data = np.load(item_invariance_folder / setup_name / str(i) / "decoding_accuracy.npy")
                        data_single_model["item_invariance_accuracy"] = item_invariance_data
                        item_invariance_cross_phase_data = np.load(item_invariance_folder / setup_name / str(i) / "cross_phase_decoding_accuracy.npy")
                        data_single_model["item_invariance_cross_phase_accuracy"] = item_invariance_cross_phase_data
                        data_single_model["item_invariance_cross_phase_accuracy_last"] = np.min(item_invariance_cross_phase_data)
                    else:
                        data_single_model["item_invariance_accuracy"] = None
                        data_single_model["item_invariance_cross_phase_accuracy"] = None
                        data_single_model["item_invariance_cross_phase_accuracy_last"] = None
                    if os.path.exists(time_invariance_folder / setup_name / str(i) / "decoding_accuracy.npy"):
                        time_invariance_data = np.load(time_invariance_folder / setup_name / str(i) / "decoding_accuracy.npy")
                        data_single_model["time_invariance_accuracy"] = time_invariance_data
                        time_invariance_cross_phase_data = np.load(time_invariance_folder / setup_name / str(i) / "cross_phase_decoding_accuracy.npy")
                        data_single_model["time_invariance_cross_phase_accuracy"] = time_invariance_cross_phase_data
                        data_single_model["time_invariance_cross_phase_accuracy_last"] = np.min(time_invariance_cross_phase_data)
                    else:
                        data_single_model["time_invariance_accuracy"] = None
                        data_single_model["time_invariance_cross_phase_accuracy"] = None
                        data_single_model["time_invariance_cross_phase_accuracy_last"] = None
                    data.append(data_single_model)

df = pd.DataFrame(data)
print(len(data))

In [ ]:
""" load vary gamma """

data_folder = Path("./experiments/VaryAllSeq8/figures/ValueMemoryGRU")
training_curve_folder = Path("./experiments/VaryAllSeq8/saved_models/ValueMemoryGRU")
perturbation_folder = Path("./experiments/VaryAllSeq8/figures/perturbation/ValueMemoryGRU")
item_invariance_folder = Path("./experiments/VaryAllSeq8/figures/item_invariant/ValueMemoryGRU")
time_invariance_folder = Path("./experiments/VaryAllSeq8/figures/time_invariant/ValueMemoryGRU")
seq_len = 8

exp = "main"

for gamma_name, temporal_discount_factor in zip(gamma_names, temporal_discount_factors):
    for noise_name, noise_level in zip(noise_names, noise_levels):
        setup_name = "setup_gamma{}_noise{}".format(gamma_name, noise_name)

        parameters = {
            "temporal_discount_factor": temporal_discount_factor,
            "noise_level": noise_level,
            "seq_len": seq_len,
            "wm_noise": 0.04
        }
        
        training_data_length = 200
        for i in range(20):
            data_path = data_folder / setup_name / str(i)
            training_data_path = training_curve_folder / (setup_name + "-" + str(i))
            if os.path.exists(data_path/"contiguity_effect.csv") and os.path.exists(data_path/"cross_acc.npy"):
                data_single_model = load_data(data_path, training_data_path, i, exp, parameters, training_data_length)
                if data_single_model:
                    data_single_model["pretrained"] = False
                    perturbation_data = np.load(perturbation_folder / setup_name / str(i) / "perturbation_accuracies.npy")
                    data_single_model["perturbation_accuracy"] = perturbation_data
                    if os.path.exists(item_invariance_folder / setup_name / str(i) / "decoding_accuracy.npy"):
                        item_invariance_data = np.load(item_invariance_folder / setup_name / str(i) / "decoding_accuracy.npy")
                        data_single_model["item_invariance_accuracy"] = item_invariance_data
                        # data_single_model["item_invariance_cross_phase_accuracy"] = None
                        # data_single_model["item_invariance_cross_phase_accuracy_last"] = None
                        item_invariance_cross_phase_data = np.load(item_invariance_folder / setup_name / str(i) / "cross_phase_decoding_accuracy.npy")
                        data_single_model["item_invariance_cross_phase_accuracy"] = item_invariance_cross_phase_data
                        data_single_model["item_invariance_cross_phase_accuracy_last"] = np.min(item_invariance_cross_phase_data)
                    else:
                        data_single_model["item_invariance_accuracy"] = None
                        data_single_model["item_invariance_cross_phase_accuracy"] = None
                        data_single_model["item_invariance_cross_phase_accuracy_last"] = None
                    if os.path.exists(time_invariance_folder / setup_name / str(i) / "decoding_accuracy.npy"):
                        time_invariance_data = np.load(time_invariance_folder / setup_name / str(i) / "decoding_accuracy.npy")
                        data_single_model["time_invariance_accuracy"] = time_invariance_data
                        # data_single_model["time_invariance_cross_phase_accuracy"] = None
                        # data_single_model["time_invariance_cross_phase_accuracy_last"] = None
                        time_invariance_cross_phase_data = np.load(time_invariance_folder / setup_name / str(i) / "cross_phase_decoding_accuracy.npy")
                        data_single_model["time_invariance_cross_phase_accuracy"] = time_invariance_cross_phase_data
                        data_single_model["time_invariance_cross_phase_accuracy_last"] = np.min(time_invariance_cross_phase_data)
                    else:
                        data_single_model["time_invariance_accuracy"] = None
                        data_single_model["time_invariance_cross_phase_accuracy"] = None
                        data_single_model["time_invariance_cross_phase_accuracy_last"] = None
                    data.append(data_single_model)

df = pd.DataFrame(data)
print(len(data))

In [ ]:
""" load vary gamma """

# data = []

data_folder = Path("./experiments/VaryAllSeq8NoNoise/figures/ValueMemoryGRU")
training_curve_folder = Path("./experiments/VaryAllSeq8NoNoise/saved_models/ValueMemoryGRU")
perturbation_folder = Path("./experiments/VaryAllSeq8NoNoise/figures/perturbation/ValueMemoryGRU")
item_invariance_folder = Path("./experiments/VaryAllSeq8NoNoise/figures/item_invariant/ValueMemoryGRU")
time_invariance_folder = Path("./experiments/VaryAllSeq8NoNoise/figures/time_invariant/ValueMemoryGRU")
seq_len = 8

exp = "main"

for gamma_name, temporal_discount_factor in zip(gamma_names, temporal_discount_factors):
    for noise_name, noise_level in zip(noise_names, noise_levels):
        setup_name = "setup_gamma{}_noise{}".format(gamma_name, noise_name)

        parameters = {
            "temporal_discount_factor": temporal_discount_factor,
            "noise_level": noise_level,
            "seq_len": seq_len,
            "wm_noise": 0.0
        }
        
        training_data_length = 200
        for i in range(20):
            data_path = data_folder / setup_name / str(i)
            training_data_path = training_curve_folder / (setup_name + "-" + str(i))
            if os.path.exists(data_path/"contiguity_effect.csv") and os.path.exists(data_path/"cross_acc.npy"):
                data_single_model = load_data(data_path, training_data_path, i, exp, parameters, training_data_length)
                if data_single_model:
                    data_single_model["pretrained"] = False
                    perturbation_data = np.load(perturbation_folder / setup_name / str(i) / "perturbation_accuracies.npy")
                    data_single_model["perturbation_accuracy"] = perturbation_data
                    if os.path.exists(item_invariance_folder / setup_name / str(i) / "decoding_accuracy.npy"):
                        item_invariance_data = np.load(item_invariance_folder / setup_name / str(i) / "decoding_accuracy.npy")
                        data_single_model["item_invariance_accuracy"] = item_invariance_data
                        # data_single_model["item_invariance_cross_phase_accuracy"] = None
                        # data_single_model["item_invariance_cross_phase_accuracy_last"] = None
                        item_invariance_cross_phase_data = np.load(item_invariance_folder / setup_name / str(i) / "cross_phase_decoding_accuracy.npy")
                        data_single_model["item_invariance_cross_phase_accuracy"] = item_invariance_cross_phase_data
                        data_single_model["item_invariance_cross_phase_accuracy_last"] = np.min(item_invariance_cross_phase_data)
                    else:
                        data_single_model["item_invariance_accuracy"] = None
                        data_single_model["item_invariance_cross_phase_accuracy"] = None
                        data_single_model["item_invariance_cross_phase_accuracy_last"] = None
                    if os.path.exists(time_invariance_folder / setup_name / str(i) / "decoding_accuracy.npy"):
                        time_invariance_data = np.load(time_invariance_folder / setup_name / str(i) / "decoding_accuracy.npy")
                        data_single_model["time_invariance_accuracy"] = time_invariance_data
                        # data_single_model["time_invariance_cross_phase_accuracy"] = None
                        # data_single_model["time_invariance_cross_phase_accuracy_last"] = None
                        time_invariance_cross_phase_data = np.load(time_invariance_folder / setup_name / str(i) / "cross_phase_decoding_accuracy.npy")
                        data_single_model["time_invariance_cross_phase_accuracy"] = time_invariance_cross_phase_data
                        data_single_model["time_invariance_cross_phase_accuracy_last"] = np.min(time_invariance_cross_phase_data)
                    else:
                        data_single_model["time_invariance_accuracy"] = None
                        data_single_model["time_invariance_cross_phase_accuracy"] = None
                        data_single_model["time_invariance_cross_phase_accuracy_last"] = None
                    data.append(data_single_model)

df = pd.DataFrame(data)
print(len(data))

In [ ]:
data_reservoir = []

exp = "reservoir"

data_folder = Path("./experiments/Reservoir/figures/ValueMemoryGRU")
training_curve_folder = Path("./experiments/Reservoir/saved_models/ValueMemoryGRU")
perturbation_folder = Path("./experiments/Reservoir/figures/perturbation/ValueMemoryGRU")
item_invariance_folder = Path("./experiments/Reservoir/figures/item_invariant/ValueMemoryGRU")
time_invariance_folder = Path("./experiments/Reservoir/figures/time_invariant/ValueMemoryGRU")
seq_len = 8

setups = ["setup_reservoir", "setup_reservoir_nopretrain", "setup_reservoir_nopretrain_initnoise"]

parameters = {
    "temporal_discount_factor": temporal_discount_factor,
    "noise_level": noise_level,
    "seq_len": seq_len,
}

for setup in setups:
    for i in range(5):
        data_path = data_folder / setup / str(i)
        training_data_path = training_curve_folder / (setup + "-" + str(i))
        if os.path.exists(data_path/"contiguity_effect.csv") and os.path.exists(data_path/"cross_acc.npy"):
            data_single_model = load_data(data_path, training_data_path, i, exp, parameters, training_data_length)
            if data_single_model:
                data_single_model["pretrained"] = False
                # perturbation_data = np.load(perturbation_folder / setup_name / str(i) / "perturbation_accuracies.npy")
                # data_single_model["perturbation_accuracy"] = perturbation_data
                if os.path.exists(item_invariance_folder / setup / str(i) / "decoding_accuracy.npy"):
                    item_invariance_data = np.load(item_invariance_folder / setup / str(i) / "decoding_accuracy.npy")
                    data_single_model["item_invariance_accuracy"] = item_invariance_data
                    item_invariance_cross_phase_data = np.load(item_invariance_folder / setup / str(i) / "cross_phase_decoding_accuracy.npy")
                    data_single_model["item_invariance_cross_phase_accuracy"] = item_invariance_cross_phase_data
                else:
                    data_single_model["item_invariance_accuracy"] = None
                    data_single_model["item_invariance_cross_phase_accuracy"] = None
                if os.path.exists(time_invariance_folder / setup / str(i) / "decoding_accuracy.npy"):
                    time_invariance_data = np.load(time_invariance_folder / setup / str(i) / "decoding_accuracy.npy")
                    data_single_model["time_invariance_accuracy"] = time_invariance_data
                    time_invariance_cross_phase_data = np.load(time_invariance_folder / setup / str(i) / "cross_phase_decoding_accuracy.npy")
                    data_single_model["time_invariance_cross_phase_accuracy"] = time_invariance_cross_phase_data
                else:
                    data_single_model["time_invariance_accuracy"] = None
                    data_single_model["time_invariance_cross_phase_accuracy"] = None
                data_reservoir.append(data_single_model)

df_reservoir = pd.DataFrame(data_reservoir)
print(len(df_reservoir))

In [ ]:
data_tcm = []

exp = "tcm"

data_folder = Path("./experiments/TCM/figures/TCM")
training_curve_folder = Path("./experiments/TCM/saved_models/TCM")
perturbation_folder = Path("./experiments/TCM/figures/perturbation/TCM")
seq_len = 8

tcm_gamma_names = ["0", "05", "1"]

for gamma_name in tcm_gamma_names:
    setup_name = "setup_gamma{}_optimal".format(gamma_name) # "setup_gamma" + gamma_name
    for i in range(5):
        data_path = data_folder / setup_name / str(i)
        training_data_path = training_curve_folder / (setup_name + "-" + str(i))
        if os.path.exists(data_path/"contiguity_effect.csv") and os.path.exists(data_path/"cross_acc.npy"):
            data_single_model = load_data(data_path, training_data_path, i, exp, parameters, training_data_length)
            if data_single_model:
                data_single_model["pretrained"] = False
                perturbation_data = np.load(perturbation_folder / setup_name / str(i) / "perturbation_accuracies.npy")
                data_single_model["perturbation_accuracy"] = perturbation_data
                data_tcm.append(data_single_model)

df_tcm = pd.DataFrame(data_tcm)
print(len(df_tcm))

In [ ]:
data_perf = []

exp = "performance"

data_folder = Path("./experiments/Performance/figures/ValueMemoryGRU")
training_curve_folder = Path("./experiments/Performance/saved_models/ValueMemoryGRU")
perturbation_folder = Path("./experiments/Performance/figures/perturbation/ValueMemoryGRU")
seq_len = 8

exp = "tdf"

for i in range(100):
    data_path = data_folder / "setup_noise09_gamma10_largenoise" / str(i)
    training_data_path = training_curve_folder / ("setup_noise09_gamma10_largenoise" + "-" + str(i))
    if os.path.exists(data_path/"contiguity_effect.csv") and os.path.exists(data_path/"cross_acc.npy"):
        data_single_model = load_data(data_path, training_data_path, i, exp, parameters, training_data_length)
        if data_single_model:
            data_single_model["pretrained"] = False
            data_perf.append(data_single_model)

df_perf = pd.DataFrame(data_perf)
print(len(df_perf))

In [ ]:
data_perf_nonoise = []

exp = "performance"

data_folder = Path("./experiments/Performance/figures/ValueMemoryGRU")
training_curve_folder = Path("./experiments/Performance/saved_models/ValueMemoryGRU")
perturbation_folder = Path("./experiments/Performance/figures/perturbation/ValueMemoryGRU")
seq_len = 8

exp = "tdf"

parameters = {
    "temporal_discount_factor": 1.0,
    "noise_level": 0.04,
    "seq_len": seq_len,
}

training_data_length = 200

for i in range(100):
    data_path = data_folder / "setup_noise09_gamma10" / str(i)
    training_data_path = training_curve_folder / ("setup_noise09_gamma10" + "-" + str(i))
    if os.path.exists(data_path/"contiguity_effect.csv") and os.path.exists(data_path/"cross_acc.npy"):
        data_single_model = load_data(data_path, training_data_path, i, exp, parameters, training_data_length)
        if data_single_model:
            data_single_model["pretrained"] = False
            data_perf_nonoise.append(data_single_model)

df_perf_nonoise = pd.DataFrame(data_perf_nonoise)
print(len(df_perf_nonoise))

In [ ]:
data_ctx = []

data_folder = Path("./experiments/ExtraObs/EnvCxt/figures/ValueMemoryGRU")
training_curve_folder = Path("./experiments/ExtraObs/EnvCxt/saved_models/ValueMemoryGRU")


setup_base_name = "setup_gamma1_flush09_gaussiannoise"
# setup_base_name = "setup_gamma1_flush09_dim41_gaussiannoise"
noise_stds = [0, 0.2, 0.4, 0.8]

setup_names = []
for noise_std in noise_stds:
    setup_names.append(setup_base_name + str(noise_std).replace(".", ""))



seq_len = 8

exp = "ctx"

for setup_name, noise_std in zip(setup_names, noise_stds):
    parameters = {
        "noise_std": noise_std,
        "seq_len": seq_len,
        "gamma": 1.0,
        "flush_noise": 0.9
    }
    
    training_data_length = 200
    for i in range(20):
        data_path = data_folder / setup_name / str(i)
        training_data_path = training_curve_folder / (setup_name + "-" + str(i))
        if os.path.exists(data_path/"contiguity_effect.csv") and os.path.exists(data_path/"cross_acc.npy"):
            data_single_model = load_data(data_path, training_data_path, i, exp, parameters, training_data_length)
            if data_single_model:
                data_single_model["pretrained"] = False
                data_ctx.append(data_single_model)

df_ctx = pd.DataFrame(data_ctx)
print(len(df_ctx))

In [ ]:
data_samediff = []

data_folder = Path("./experiments/ExtraObs/RecallCtx/figures/ValueMemoryGRU")
training_curve_folder = Path("./experiments/ExtraObs/RecallCtx/saved_models/ValueMemoryGRU")

setup_names = ["setup_gaussiannoise04_dim41_gamma09_flush09_same2", "setup_gaussiannoise04_dim41_gamma09_flush09_diff2"]
noise_types = ["same", "diff"]

seq_len = 8

exp = "ctx"

for k, setup_name, noise_type in zip(range(len(setup_names)), setup_names, noise_types):
    parameters = {
        "noise_type": noise_type,
        "seq_len": seq_len,
        "test_noise_type": "same",
        "mark": (k+1)*2
    }
    
    training_data_length = 200
    for i in range(20):
        data_path = data_folder / (setup_name+"_testsame") / str(i)
        training_data_path = training_curve_folder / (setup_name + "-" + str(i))
        if os.path.exists(data_path/"contiguity_effect.csv") and os.path.exists(data_path/"cross_acc.npy"):
            data_single_model = load_data(data_path, training_data_path, i, exp, parameters, training_data_length)
            if data_single_model:
                data_single_model["pretrained"] = False
                data_samediff.append(data_single_model)

for k, setup_name, noise_type in zip(range(len(setup_names)), setup_names, noise_types):
    parameters = {
        "noise_type": noise_type,
        "seq_len": seq_len,
        "test_noise_type": "diff",
        "mark": (k+1)*2+1
    }
    
    training_data_length = 200
    for i in range(20):
        data_path = data_folder / (setup_name+"_testdiff")/ str(i)
        training_data_path = training_curve_folder / (setup_name + "-" + str(i))
        if os.path.exists(data_path/"contiguity_effect.csv") and os.path.exists(data_path/"cross_acc.npy"):
            data_single_model = load_data(data_path, training_data_path, i, exp, parameters, training_data_length)
            if data_single_model:
                data_single_model["pretrained"] = False
                data_samediff.append(data_single_model)


setup_name = "setup_gaussiannoise04_dim41_gamma09_flush09_fixed2"
noise_type = "fixed"
parameters = {
    "noise_type": noise_type,
    "seq_len": seq_len,
    "test_noise_type": "fixed",
    "mark": 0
}

training_data_length = 200
for i in range(20):
    data_path = data_folder / (setup_name) / str(i)
    training_data_path = training_curve_folder / (setup_name + "-" + str(i))
    if os.path.exists(data_path/"contiguity_effect.csv") and os.path.exists(data_path/"cross_acc.npy"):
        data_single_model = load_data(data_path, training_data_path, i, exp, parameters, training_data_length)
        if data_single_model:
            data_single_model["pretrained"] = False
            data_samediff.append(data_single_model)


data_folder = Path("./experiments/ExtraObs/EnvCxt/figures/ValueMemoryGRU")
training_curve_folder = Path("./experiments/ExtraObs/EnvCxt/saved_models/ValueMemoryGRU")
setup_name = "setup_gamma09_flush09_gaussiannoise0"
parameters = {
    "noise_type": "none",
    "seq_len": seq_len,
    "test_noise_type": "none",
    "mark": 1
}

for i in range(20):
    data_path = data_folder / (setup_name) / str(i)
    training_data_path = training_curve_folder / (setup_name + "-" + str(i))
    if os.path.exists(data_path/"contiguity_effect.csv") and os.path.exists(data_path/"cross_acc.npy"):
        data_single_model = load_data(data_path, training_data_path, i, exp, parameters, training_data_length)
        if data_single_model:
            data_single_model["pretrained"] = False
            data_samediff.append(data_single_model)


df_samediff_raw = pd.DataFrame(data_samediff)
print(len(df_samediff_raw))

In [ ]:
data_semantic = []

data_folder = Path("./experiments/Semantic/figures/ValueMemoryGRU")
training_curve_folder = Path("./experiments/Semantic/saved_models/ValueMemoryGRU")
semantic_folder = Path("./experiments/Semantic/figures/semantic/ValueMemoryGRU")

setup_name = "setup_extra_hierarchy_amp"
# semantic_amp = [0.3, 0.4, 0.5]
semantic_amp = [0, 0.3, 0.5]
setup_names = [setup_name + str(amp).replace(".", "") for amp in semantic_amp]
print(setup_names)



seq_len = 8

exp = "semantic"

for i, setup_name in enumerate(setup_names):
    parameters = {
        "semantic_amp": semantic_amp[i],
    }
    
    training_data_length = 200
    for i in range(20):
        data_path = data_folder / setup_name / str(i)
        training_data_path = training_curve_folder / (setup_name + "-" + str(i))
        if os.path.exists(data_path/"contiguity_effect.csv") and os.path.exists(data_path/"cross_acc.npy"):
            data_single_model = load_data(data_path, training_data_path, i, exp, parameters, training_data_length)
            if data_single_model:
                data_single_model["pretrained"] = False
                semantic_contiguity_data = np.load(semantic_folder / setup_name / str(i) / "data" / "semantic_contiguity_results.npy")
                data_single_model["semantic_contiguity"] = semantic_contiguity_data
                semantic_contiguity_baseline_data = np.load(semantic_folder / setup_name / str(i) / "data" / "semantic_contiguity_baseline.npy")
                data_single_model["semantic_contiguity_baseline"] = semantic_contiguity_baseline_data
                data_semantic.append(data_single_model)

df_semantic = pd.DataFrame(data_semantic)
print(len(df_semantic))


### save data

In [ ]:
df.to_pickle("./data/df_models.pkl")

In [ ]:
df_reservoir.to_pickle("./data/df_reservoir.pkl")
df_tcm.to_pickle("./data/df_tcm.pkl")
df_perf.to_pickle("./data/df_perf.pkl")

In [ ]:
df_perf_nonoise.to_pickle("./data/df_perf_nonoise.pkl")

In [ ]:
df_ctx.to_pickle("./data/df_ctx.pkl")
df_samediff_raw.to_pickle("./data/df_samediff.pkl")
df_semantic.to_pickle("./data/df_semantic.pkl")

### test load data

In [ ]:
df = pd.read_pickle("./data/df_models.pkl")

In [ ]:
# training_accuracy_by_tdf = df_tdf[(df_tdf["pretrained"] == True) & (df_tdf["eta"] == 0.01)].groupby("temporal_discount_factor")["training_accuracy"].mean()
df_filtered = df[(df["accuracy"] > 0.7) & (df["noise_level"] == 1.0)]
training_accuracy_by_tdf = df_filtered.groupby("temporal_discount_factor")["training_accuracy"].mean()

colors = plt.cm.viridis(np.linspace(0, 1, len(training_accuracy_by_tdf)))
for i, (tdf, acc) in enumerate(training_accuracy_by_tdf.items()):
    plt.plot(acc[:200], label=f"TDF = {tdf:.1f}", color=colors[i])
    # std_dev = np.std(np.stack(df_tdf_filtered[df_tdf_filtered["temporal_discount_factor"] == tdf]["training_accuracy"]), axis=0)
    # plt.fill_between(range(len(acc[:200])), acc[:200] - std_dev, acc[:200] + std_dev, color=colors[i], alpha=0.2)
    
# set the legend to the right of the plot
ax = plt.gca()
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False)
plt.xlabel("Training epochs (1000)")
plt.ylabel("Training accuracy")
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.show()


In [ ]:
def plot_crp_by_group(data, group_values, group_title, colors):
    plt.figure(figsize=(3.5, 3), dpi=180)

    print(data.shape[1])

    mid_point = data.shape[1] // 2 + 1

    print(mid_point)

    for i in range(len(group_values)):
        plt.plot(np.arange(-mid_point+1, 0), data[i, :mid_point-1], label=group_values[i], color=colors[i])
        plt.plot(np.arange(1, mid_point), data[i, mid_point:], color=colors[i])
    
    plt.xlabel("Lag")
    plt.ylabel("Conditional\nrecall probability")
    plt.legend(
        title=group_title, 
        bbox_to_anchor=(1.0, 1.0), 
        loc="upper left", 
        borderaxespad=0, 
        frameon=False
    )
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.show()

In [ ]:
crp_by_noise_std = df.groupby("temporal_discount_factor")["crp_curve"]
# print(crp_by_noise_std.head())
mean_crp_by_noise_std = []
for name, group in crp_by_noise_std:
    crp_arrays = np.array(group.tolist())
    # print(crp_arrays.shape)
    mean_crp = np.mean(np.stack(crp_arrays, axis=0), axis=0)
    mean_crp_by_noise_std.append(mean_crp)
mean_crp_by_noise_std = np.array(mean_crp_by_noise_std)

viridis = plt.get_cmap('viridis', lut=len(gamma_names))
colors = [viridis(i) for i in range(len(gamma_names))]

# print(mean_crp_by_noise_std)
plot_crp_by_group(mean_crp_by_noise_std, temporal_discount_factors, "Discount factor", colors)



In [ ]:
print(df["temporal_factor"].min(), df["temporal_factor"].max())